# Training: the classifiers of one encoder, a pilot of two runs and then the grid

Run at a fixed commit on a Colab runtime with a T4 GPU, after the features
of the encoder exist (`notebooks/03_extract_features.ipynb`).

The cell "What this run does" has two values. `ENCODER` is the encoder whose
classifiers are trained. `RUN_GRID` is false as saved: the notebook then
trains the two pilot runs, checks them, prints the estimated time of the
grid and stops. With `RUN_GRID = True` it goes on to the grid.

- `ENCODER = "hubert-base"` (as saved): the 480 runs of the analysis plan on
  the unpadded features, with the training function, the seeds and the
  checkpoint rule of WavLM Base. Every model is scored on the development
  set, the evaluation set and In-the-Wild in every test condition of its
  training condition. There are no original scores to compare with and no
  padded runs. Instead, three things are checked: every run finished, every
  score set has the ids and the labels of the file list in the same order,
  and every row of `runs.csv` records the encoder, the checkpoint revision
  of the registry and a commit without uncommitted changes.
- `ENCODER = "wavlm-base"`: the reproduction of WavLM Base from the migrated
  features (gate A2). Every score array is compared with the scores of the
  original analysis: identical ids and labels, a largest absolute score
  difference of at most `verify.SCORE_TOLERANCE` (1e-4) and the same best
  epoch; `exactly_zero` says whether the difference is 0. The padded
  comparison (10 runs) follows the grid.

**This notebook computes no result.** It writes the score files and
`runs.csv` under `runs/` of the new root and check tables under `checks/`.
The only metric in it is the development-set EER that selects the checkpoint
of a run (`best_dev_eer_roc_corners`); it is never reported. The analysis
tables are computed by `notebooks/08_analysis.ipynb`. The features are only
read.

1. the plan: the runs of the experiment configuration, and how many are finished;
2. **pilot**: layer 4, both statistics, seed 42, untrimmed and trimmed at 40 dB (two runs), trained and checked. The notebook stops here if the pilot fails;
3. the number of runs and the estimated time on this runtime, from the pilot;
4. with `RUN_GRID`: the grid of the unpadded features (480 runs), then the check of all its runs;
5. with `RUN_GRID`, WavLM Base only: the runs of the padded comparison and their check;
6. summary (hard stop if any row fails).

The grid can be interrupted. A run is finished when its score file and its
row of `runs.csv` exist on Drive; after a restart the notebook is run again
from the top and every finished run is skipped. Only the local feature cache
is built again, because it lives on the local disk of the runtime.

In [ ]:
from google.colab import drive, userdata
import os

drive.mount("/content/drive")
assert os.path.ismount("/content/drive"), "Google Drive is not mounted"

### Repository at a fixed commit

Set `COMMIT` to the commit of this repository that is being checked: the
full hash or at least its first 7 characters. The repository is public and
is cloned without a token. If the Colab secret `GITHUB_TOKEN` exists and
this notebook has access to it, the clone uses it (a repository that is not
public needs one); the token is not kept in the clone and is replaced by
`***` in everything git prints.

In [ ]:
import subprocess

REPOSITORY = "Nooralabd/silence-shortcut-wavlm-layers"
COMMIT = ""          # the commit to run: the full hash or at least its first 7 characters
WORKDIR = "/content/silence-shortcut-wavlm-layers"

assert len(COMMIT) >= 7 and set(COMMIT) <= set("0123456789abcdef"), \
    "set COMMIT to the commit that is being checked (full hash or at least its first 7 characters)"
try:
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:    # the secret does not exist, or this notebook has no access to it
    token = ""       # a public repository is cloned without a token


def git(*arguments):
    """Run git and print what it prints; the token is never printed."""
    result = subprocess.run(["git", *arguments], capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    if token:
        output = output.replace(token, "***")
    if output:
        print(output)
    assert result.returncode == 0, "git failed: see the lines above"
    return output


def clone(repository, folder):
    """Clone a repository, or fetch into an existing clone; a token, if there is one, is not kept in it."""
    address = f"https://{token}@github.com/{repository}.git" if token else f"https://github.com/{repository}.git"
    if not os.path.isdir(folder):
        git("clone", "-q", address, folder)
        git("-C", folder, "remote", "set-url", "origin", f"https://github.com/{repository}.git")
    else:
        git("-C", folder, "fetch", "-q", address, "+refs/heads/*:refs/remotes/origin/*")


clone(REPOSITORY, WORKDIR)
git("-C", WORKDIR, "checkout", "-q", COMMIT)
git("-C", WORKDIR, "rev-parse", "HEAD")
%cd {WORKDIR}

In [ ]:
# Installation. torch, torchaudio and torchcodec are the builds that extracted
# the original features: the pinned versions for CUDA 13.0. If anything pinned
# is missing or has another version, everything is installed in one pass and
# the cell always stops for a restart: a kernel never continues after pip has
# run in it (the runtime has imported numpy before this cell, and a module
# that is already loaded stays loaded when pip replaces it on disk).
# On the run after the restart nothing is installed: the cell checks the
# installed versions and the versions of the modules that the kernel loads.
import importlib
import importlib.metadata
import re
import sys

TORCH_INDEX = "https://download.pytorch.org/whl/cu130"
TORCH_FAMILY = ("torch", "torchaudio", "torchcodec")          # installed from TORCH_INDEX
INSTALLED_ONCE = os.path.join(os.path.dirname(WORKDIR), "installed_once.txt")
OTHERS = os.path.join(os.path.dirname(WORKDIR), "requirements_without_torch.txt")
CONSTRAINT = os.path.join(os.path.dirname(WORKDIR), "torch_constraint.txt")

with open(os.path.join(WORKDIR, "requirements.txt"), encoding="utf-8") as fh:
    requirements = [line.strip() for line in fh if line.strip() and not line.lstrip().startswith("#")]
pins = dict(line.split("==") for line in requirements if "==" in line)
BUILDS = {name: pins[name] + "+" + TORCH_INDEX.rsplit("/", 1)[1] for name in TORCH_FAMILY}


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def not_as_pinned():
    """{package: (installed, wanted)} for everything that is missing or has another version."""
    found = {}
    for line in requirements:
        name = re.split(r"[=<>!~ ]", line, maxsplit=1)[0]
        have, want = installed(name), BUILDS.get(name, pins.get(name))
        same = have is not None and (want is None or (have if name in BUILDS else have.split("+")[0]) == want)
        if not same:
            found[name] = (have, want)
    if installed("ssl-silence") is None:
        found["ssl-silence"] = (None, "this repository")
    return found


different = not_as_pinned()
if different:
    assert not os.path.exists(INSTALLED_ONCE), (
        f"the libraries were installed once already and these are still not as pinned: {different}. "
        "Stop: do not restart again")
    imported = sorted(name for name in (*TORCH_FAMILY, "transformers", "ssl_silence") if name in sys.modules)
    assert not imported, (f"{imported} were imported before the installation: "
                          "Runtime > Restart session, then run the notebook again from the top")
    with open(OTHERS, "w", encoding="utf-8") as fh:
        for line in requirements:
            if re.split(r"[=<>!~ ]", line, maxsplit=1)[0] not in TORCH_FAMILY:
                print(line, file=fh)
    with open(CONSTRAINT, "w", encoding="utf-8") as fh:
        for name in TORCH_FAMILY:
            print(f"{name}=={BUILDS[name]}", file=fh)
    # The build is part of the pin (torch==2.11.0+cu130): for a pin without it pip
    # keeps another build of the same version that the runtime ships (+cpu).
    FAMILY_PINS = " ".join(f"{name}=={BUILDS[name]}" for name in TORCH_FAMILY)
    # torchvision of the runtime is built for its own torch and breaks imports
    # with another one; this package does not use it.
    !pip uninstall -q -y torchvision
    !pip install -q {FAMILY_PINS} --index-url {TORCH_INDEX}
    !pip install -q -r {OTHERS} -c {CONSTRAINT}
    !pip install -q --no-deps -e .
    importlib.invalidate_caches()
    # Before the restart: everything on disk is as pinned, so the next run installs nothing.
    different = not_as_pinned()
    assert not different, f"pip did not install these as pinned: {different}: stop"
    with open(INSTALLED_ONCE, "w", encoding="utf-8") as fh:
        print("the pinned libraries were installed in this runtime", file=fh)
    # pip has run in this kernel: it always stops here.
    raise RuntimeError("The pinned libraries were installed. "
                       "Runtime > Restart session, then run the notebook again from the top")

# Nothing was installed in this run. The modules that this kernel loads must
# be the pinned versions too, not only the files on disk.
LOADED = {"numpy": "numpy", "scipy": "scipy", "pandas": "pandas", "scikit-learn": "sklearn",
          "torch": "torch", "torchaudio": "torchaudio", "transformers": "transformers"}
for name, module in LOADED.items():
    loaded = importlib.import_module(module).__version__
    assert loaded.split("+")[0] == pins[name], (
        f"{module} {loaded} is loaded in this kernel and the pin is {pins[name]}: it was imported before pip "
        "replaced it. Runtime > Restart session, then run the notebook again from the top")

import torch
import torchaudio
import transformers

print("loaded:", {name: sys.modules[module].__version__ for name, module in LOADED.items()})
print("torch", torch.__version__, "| CUDA of the build:", torch.version.cuda)
print("torchaudio", torchaudio.__version__, "| torchcodec", installed("torchcodec"),
      "| transformers", transformers.__version__)
if os.path.join(WORKDIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(WORKDIR, "src"))   # a running kernel does not see a new editable install

In [ ]:
import subprocess
import sys

import pandas as pd

from ssl_silence import config as cfg
from ssl_silence import provenance, verify

pd.set_option("display.max_rows", 200, "display.width", 200)
HEAD = provenance.git_commit()     # the full hash, with "-dirty" if a tracked file was changed
assert HEAD.startswith(COMMIT) and not HEAD.endswith("-dirty"), (HEAD, COMMIT)
print("commit:", HEAD)
print("root:", cfg.ROOT)

In [ ]:
# The quota of the Google account is read from the Drive API (this asks for
# access to the account); the mounted folder does not report it.
QUOTA = cfg.drive_quota()
print("Google Drive: {usage_gb:.1f} GB used of {limit_gb:.1f} GB, {free_gb:.1f} GB free".format(**QUOTA),
      "(GB = 10^9 bytes; required: %d GB free)" % cfg.MIN_FREE_GB)
cfg.require_free_space(quota=QUOTA)
print("local disk of the runtime: %.1f GB free" % cfg.local_free_space_gb())

### Mapping file

`legacy_map.yaml` names the original folders, files and labels. It is kept on
Drive only, at `<root>/migration/legacy_map.yaml`, and must have no empty
value (`verify.load_map` stops otherwise).

In [ ]:
MAP_PATH = os.path.join(cfg.migration_dir(), "legacy_map.yaml")
assert os.path.exists(MAP_PATH), f"upload legacy_map.yaml to {MAP_PATH}"
legacy = verify.load_map(MAP_PATH)
FORBIDDEN = legacy["forbidden"]
CHECKS = cfg.checks_dir()
checks = {}


def record(name, table, stop=True):
    """Write a check table to Drive, show it, and stop if a row fails."""
    path = verify.write_check(table, name, CHECKS, FORBIDDEN)
    checks[name] = table
    display(table)
    if stop and "ok" in table.columns:
        verify.require_ok(table, name)
    return path

### What this run does

Leave `RUN_GRID` false for the pilot and the time estimate. Set it to true
to train the grid.

In [ ]:
ENCODER = "hubert-base"    # "hubert-base": the runs of the analysis plan; "wavlm-base": the reproduction (gate A2)
RUN_GRID = False           # False: the pilot and the time estimate only; True: then the grid

REFERENCE = "wavlm-base"                   # the encoder of the original analysis
ORIGINAL_SCORES = ENCODER == REFERENCE     # original scores exist for this encoder only
assert ENCODER in cfg.experiments()["experiments"], \
    f"ENCODER is {ENCODER!r}: set it to one of {list(cfg.experiments()['experiments'])}"
KINDS = [kind for kind in cfg.EXTRACTIONS if kind in cfg.experiments()["experiments"][ENCODER]]


def table_name(kind, part):
    """Name of a check table; the tables of the reference encoder keep their names."""
    return f"{kind}_{part}" if ORIGINAL_SCORES else f"{kind}_{ENCODER}_{part}"


print("this run:", ENCODER, "| extractions:", KINDS, "| grid:", RUN_GRID,
      "| checked against:", "the original scores" if ORIGINAL_SCORES else "the file lists and the run records")

### Plan

One model per training condition, layer, pooling and seed
(`src/ssl_silence/configs/experiments.yaml`). The features are read through
a local cache with one array per feature key, so that a layer is read
without reading every shard. Building the cache reads every shard from Drive
once per runtime.

In [ ]:
import torch

from ssl_silence import datasets, training

assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > T4 GPU, then run again from the top"
DEVICE = "cuda"
planned = {kind: training.plan_runs(ENCODER, kind) for kind in KINDS}
RUNS = {kind: cfg.runs_dir(ENCODER, kind) for kind in KINDS}
pilot = training.pilot_runs(planned["unpadded"])
PILOT_IDS = [run.spec.run_id for run in pilot]
for kind in KINDS:
    print(kind, "| runs folder:", RUNS[kind])
    display(training.grid_report(planned[kind], RUNS[kind]))
print("pilot:", PILOT_IDS)

entry = cfg.encoder(ENCODER)
print("encoder:", ENCODER, "|", entry["hf_id"], "| revision", entry["revision"])
n_files = sum(cfg.ASVSPOOF_SPLIT_SIZES.values()) + cfg.ITW_SIZE
cache_gb = len(cfg.conditions()) * 2 * entry["n_layers"] * entry["hidden_size"] * 4 * n_files / 1e9
print("local feature cache of the unpadded features when complete: about %.0f GB" % cache_gb)
print("local disk of the runtime: %.1f GB free" % cfg.local_free_space_gb())

### File lists (without original scores)

The ids and labels that every score set must have. Only the protocol files
of ASVspoof 2019 LA and the `meta.csv` of In-the-Wild are read from the two
archives on the new root; no audio is unpacked. Every list is checked
(number of files per class, unique ids, attack ids).

In [ ]:
def check_runs(part, runs):
    """The score sets of ``runs`` against the file lists, and their run records.
    Both tables are written before the hard stop."""
    sets = training.score_set_check(runs, RUNS["unpadded"], file_lists)
    records = training.run_record_check(runs, RUNS["unpadded"], n_train=len(file_lists["train"]))
    record(table_name("score_sets", part), sets, stop=False)
    record(table_name("run_records", part), records, stop=False)
    verify.require_ok(sets, table_name("score_sets", part))
    verify.require_ok(records, table_name("run_records", part))
    return sets, records


if not ORIGINAL_SCORES:
    file_lists = datasets.archive_file_lists(cfg.la_zip(), cfg.itw_zip())
    print({split: len(file_lists[split]) for split in cfg.SPLITS})

### Pilot

The two pilot runs are trained first (or skipped if they are finished) and
checked in every condition in which they are scored: against the original
scores for WavLM Base, against the file lists and the run records
otherwise. The untrimmed model is scored in all four conditions, so the
cache of every condition is built here. **The notebook stops in this cell if
a pilot check fails**; the grid below is then not started.

In [ ]:
pilot_status = training.run_experiments(pilot, runs_dir=RUNS["unpadded"], device=DEVICE)
record(table_name("training", "pilot"), pilot_status, stop=False)
if ORIGINAL_SCORES:
    record("scores_pilot", verify.compare_runs(RUNS["unpadded"], legacy, run_ids=PILOT_IDS))
else:
    check_runs("pilot", pilot)
print("pilot passed")

### Number of runs and estimated time

`seconds_per_run` is the mean duration of the pilot runs trained in this
session on this runtime, and `hours_to_train` is that mean times the runs
that are not finished. Both are empty if the pilot runs were already
finished; the grid then prints the time left after its first run. A pilot
run uses both statistics, and the untrimmed pilot run is scored in more
conditions than most runs, so the estimate is an upper estimate rather than
an exact one. It does not include building the local feature cache, which
is done once per runtime.

In [ ]:
estimate = pd.concat([training.grid_report(planned[kind], RUNS[kind], pilot_status["seconds"]).assign(extraction=kind)
                      for kind in KINDS], ignore_index=True)
record(table_name("training", "estimate"), estimate, stop=False)
if not RUN_GRID:
    print("RUN_GRID is False: the grid is not started. For the grid set RUN_GRID = True and run again from the top;")
    print("the two pilot runs are finished and are skipped.")

In [ ]:
def finish(kind):
    """The check of the finished grid of ``kind``; the table by condition is shown before the hard stop."""
    ids = [run.spec.run_id for run in planned[kind]]
    missing = sorted(set(ids) - training.finished_run_ids(RUNS[kind]))
    assert not missing, f"{len(missing)} runs of the {kind} grid are not finished: run its grid cell again"
    runs = training.read_runs(RUNS[kind])[["run_id", "condition"]]
    if ORIGINAL_SCORES:
        scores = verify.compare_runs(RUNS[kind], legacy, run_ids=ids)
        record(f"scores_{kind}", scores, stop=False)
        display(scores.merge(runs, on="run_id").groupby("condition").agg(
            runs=("run_id", "nunique"), score_sets=("ok", "size"), failed=("ok", lambda ok: int((~ok).sum())),
            max_abs_diff=("max_abs_diff", "max"), exactly_zero=("exactly_zero", "sum"),
            best_epoch_equal=("best_epoch_equal", "sum")))
        verify.require_ok(scores, f"scores_{kind}")
    else:
        sets = training.score_set_check(planned[kind], RUNS[kind], file_lists)
        records = training.run_record_check(planned[kind], RUNS[kind], n_train=len(file_lists["train"]))
        record(table_name("score_sets", kind), sets, stop=False)
        record(table_name("run_records", kind), records, stop=False)
        display(sets.merge(runs, on="run_id").groupby("condition").agg(
            runs=("run_id", "nunique"), score_sets=("ok", "size"), failed=("ok", lambda ok: int((~ok).sum()))))
        display(records.groupby(["encoder", "encoder_revision", "git_commit"]).agg(
            runs=("run_id", "size"), failed=("ok", lambda ok: int((~ok).sum()))).reset_index())
        verify.require_ok(sets, table_name("score_sets", kind))
        verify.require_ok(records, table_name("run_records", kind))

### Grid of the unpadded features (only with `RUN_GRID`)

`MAX_NEW_RUNS = None` trains every run that is not finished. A number trains
that many runs and returns; the check below then stops because the grid is
not complete.

In [ ]:
MAX_NEW_RUNS = None
if RUN_GRID:
    status = training.run_experiments(planned["unpadded"], runs_dir=RUNS["unpadded"], device=DEVICE,
                                      max_new_runs=MAX_NEW_RUNS)
    record(table_name("training", "unpadded"), status, stop=False)
    display(training.grid_report(planned["unpadded"], RUNS["unpadded"], status["seconds"]))

Every run of the grid is checked. WavLM Base: every score set against the
original scores (a set in the training condition against the original run
scored in its own condition, a set in another condition against the original
run scored in that condition). Otherwise: every score set against the file
list of its split, and every run record. The tables by condition and by
commit are shown before the hard stop.

In [ ]:
if RUN_GRID:
    finish("unpadded")

### Padded comparison (WavLM Base only, with `RUN_GRID`)

The two reference configurations on the padded features, five seeds each.
The folder of the original padded runs in the mapping file is still marked
as to be confirmed; if it is wrong this section stops with a missing-file
error, after the unpadded scores were compared and written. The analysis
plan has no padded runs for HuBERT Base.

In [ ]:
if RUN_GRID and "padded" in KINDS:
    status = training.run_experiments(planned["padded"], runs_dir=RUNS["padded"], device=DEVICE)
    record("training_padded", status, stop=False)
    finish("padded")

### Summary

In [ ]:
tables = {name: table for name, table in checks.items() if "ok" in table.columns}
summary = verify.summarise_checks(tables)
display(summary)
print("check tables written to", CHECKS)
assert summary["ok"].all()
if not RUN_GRID:
    assert set(tables) == ({"scores_pilot"} if ORIGINAL_SCORES else
                           {table_name("score_sets", "pilot"), table_name("run_records", "pilot")})
    print("pilot finished:", ENCODER, "| the grid was not started (RUN_GRID is False)")
elif ORIGINAL_SCORES:
    assert set(tables) == {"scores_pilot", "scores_unpadded", "scores_padded"}
    print("gate A2 passed:", ENCODER, "| score sets compared:", int(summary["n_rows"].sum()))
else:
    assert set(tables) == {table_name(kind, part) for kind in ("score_sets", "run_records")
                           for part in ("pilot", "unpadded")}
    records = checks[table_name("run_records", "unpadded")]
    print("training finished:", ENCODER, "| runs:", len(records),
          "| score sets checked:", len(checks[table_name("score_sets", "unpadded")]),
          "| revision:", sorted(set(records["encoder_revision"])),
          "| commits:", sorted(set(records["git_commit"])))